# 🥈 Camada Silver

A **camada Silver** é responsável por **limpar, padronizar e estruturar** os dados provenientes da camada Bronze, garantindo consistência e qualidade antes de serem usados em análises ou modelos na camada Gold.

## 1. Renomeação de Colunas (ft_consumidores)

 Coluna na Camada Bronze -- Coluna na Camada Silver


 `customer_id`         =   **`id_consumidor`** 

 `customer_zip_code_prefix` = **`prefixo_cep`** 

 `customer_city`     =     **`cidade`** 

 `customer_state`     =  **`estado`** 

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window 

catalogo = "medalhao"
silver_db_name = "silver"

consumidores_silver = (
    spark.table('medalhao.bronze.ft_consumidores')
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado"),
        F.col("ingestion_timestamp").alias("data_ingestao_bronze")
    )

)

#  Verifiquei se haviam linhas/cartas exatamente iguais
duplicates_count = consumidores_silver.count() - consumidores_silver.dropDuplicates(['id_consumidor']).count() 
# Total de Duplicadas = Total de Linhas - Total de Linhas Únicas 
print(f"Número de linhas exatamente iguais: {duplicates_count}") # 0 linhas duplicadas

consumidores_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_consumidores")

consumidores_silver.display()

## **2.  Mapeamento e Renomeação de Colunas (`ft_pedidos`)**

 Coluna na Camada Bronze--Coluna na Camada Silver 

 `order_id` = **`id_pedido`** 

 `customer_id` = **`id_consumidor`** 

 `order_status` = **`status`**  

 `order_purchase_timestamp` = **`pedido_compra_timestamp`** 

 `order_approved_at` = **`pedido_aprovado_timestamp`** 

 `order_delivered_carrier_date` = **`pedido_carregado_timestamp`** 

 `order_delivered_customer_date` = **`pedido_entregue_timestamp`** 

 `order_estimated_delivery_date` = **`pedido_estimativa_entrega_timestamp`** 

 *(Derivada)*  **`tempo_entrega_dias`**  =  Diferença em dias entre a data de entrega e a data de compra. 

 *(Derivada)*  **`tempo_entrega_estimado_dias`** = Diferença em dias entre a data estimada e a data de compra. 

 *(Derivada)*  **`diferenca_entrega_dias`** = Diferença entre o tempo real e o tempo estimado de entrega. 

 *(Derivada)*  **`entrega_no_prazo`** = Indicador textual baseado na diferença de entrega. 

---

### **Mapeamento de Tradução da Coluna `status`**

 Status (Bronze)--Status (Silver) 

 `delivered` = **`entregue`** 

 `invoiced` = **`faturado`** 

 `shipped`  = **`enviado`** 

 `processing` = **`em processamento`** 

 `unavailable` = **`indisponível`** 

 `canceled` = **`cancelado`** 

 `created`  = **`criado`** 
 
 `approved`  = **`aprovado`** 

In [0]:

pedidos_silver = (
    spark.table('medalhao.bronze.ft_pedidos')
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),
        # Tradução de status
        F.when(F.col("order_status") == "delivered", "entregue")
         .when(F.col("order_status") == "invoiced", "faturado")
         .when(F.col("order_status") == "shipped", "enviado")
         .when(F.col("order_status") == "processing", "em processamento")
         .when(F.col("order_status") == "unavailable", "indisponível")
         .when(F.col("order_status") == "canceled", "cancelado")
         .when(F.col("order_status") == "created", "criado")
         .when(F.col("order_status") == "approved", "aprovado")
         .otherwise(F.col("order_status"))
         .alias("status"),
        
        F.col("order_purchase_timestamp").alias("pedido_compra_timestamp"),
        F.col("order_approved_at").alias("pedido_aprovado_timestamp"),
        F.col("order_delivered_carrier_date").alias("pedido_carregado_timestamp"),
        F.col("order_delivered_customer_date").alias("pedido_entregue_timestamp"),
        F.col("order_estimated_delivery_date").alias("pedido_estimativa_entrega_timestamp"),
        F.col("ingestion_timestamp").alias("data_ingestao_bronze")
    )
    # Criação das novas colunas derivadas
    .withColumn(
        "tempo_entrega_dias",
        F.datediff("pedido_entregue_timestamp", "pedido_compra_timestamp")
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff("pedido_estimativa_entrega_timestamp", "pedido_compra_timestamp")
    )
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )
    .withColumn(
        "entrega_no_prazo",
        F.when(F.col("pedido_entregue_timestamp").isNull(), "Não Entregue")
         .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
         .otherwise("Não")
    )
)

#  Verifiquei se haviam linhas/cartas exatamente iguais
duplicates_count = pedidos_silver.count() - pedidos_silver.dropDuplicates().count() 
# Total de Duplicadas = Total de Linhas - Total de Linhas Únicas 
print(f"Número de linhas exatamente iguais: {duplicates_count}") # 0 linhas iguais


pedidos_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_pedidos")

pedidos_silver.display()

## **3.  Mapeamento e Renomeação de Colunas (`ft_itens_pedidos`)**

 Coluna na Camada Bronze--Coluna na Camada Silver 

 `order_id` = **`id_pedido`**  

 `order_item_id` = **`id_item`**  

 `product_id` = **`id_produto`**  

 `seller_id` = **`id_vendedor`**  

 `price` = **`preco_BRL`**  
 
 `freight_value` = **`preco_frete`**  

In [0]:
itens_pedidos_silver = (
    spark.table('medalhao.bronze.ft_itens_pedidos')
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("price").cast("decimal(12,2)").alias("preco_BRL"),
        F.col("freight_value").cast("decimal(12,2)").alias("preco_frete"),
        F.col("ingestion_timestamp").alias("data_ingestao_bronze")
    )
)

#  Verifiquei se haviam linhas/cartas exatamente iguais
duplicates_count = itens_pedidos_silver.count() - itens_pedidos_silver.dropDuplicates().count() 
# Total de Duplicadas = Total de Linhas - Total de Linhas Únicas 
print(f"Número de linhas exatamente iguais: {duplicates_count}") # 0 linhas iguais

itens_pedidos_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_itens_pedidos")

itens_pedidos_silver.display()

## **4. Mapeamento e Renomeação de Colunas (`ft_pagamentos`)**

| Coluna na Camada Bronze | Coluna na Camada Silver |
| :---------------------- | :---------------------- |
| `order_id` | **`id_pedido`** |
| `payment_sequential` | **`codigo_pagamento`** |
| `payment_type` | **`forma_pagamento`** |
| `payment_installments` | **`parcelas`** |
| `payment_value` | **`valor_pagamento`** |

---

### **Mapeamento de Tradução da Coluna `forma_pagamento`**

A coluna `payment_type` (Bronze) foi traduzida para a coluna **`forma_pagamento`** (Silver) usando o seguinte mapeamento:

| Valor (Bronze) | Valor (Silver) |
| :---------------------- | :------------------------- |
| `credit_card` | **`Cartão de Crédito`** |
| `boleto` | **`Boleto`** |
| `voucher` | **`Voucher`** |
| `debit_card` | **`Cartão de Débito`** |
| *(Demais valores)* | **`Outro`** |


In [0]:
pagamentos_pedidos_silver = (
    spark.table('medalhao.bronze.ft_pagamentos_pedidos')
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("codigo_pagamento"),
        F.when(F.col("payment_type") == "credit_card", "Cartão de Crédito")
         .when(F.col("payment_type") == "boleto", "Boleto")
         .when(F.col("payment_type") == "voucher", "Voucher")
         .when(F.col("payment_type") == "debit_card", "Cartão de Débito")
         .otherwise("Outro")   
         .alias("forma_pagamento"),
        F.col("payment_value").cast("decimal(12,2)").alias("valor_pagamento"),
        F.col("payment_installments").alias("parcelas"),
        F.col("ingestion_timestamp").alias("data_ingestao_bronze")
    )
)

pagamentos_pedidos_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_pagamentos_pedidos")

pagamentos_pedidos_silver.display()

## 5. Mapeamento e Renomeação de Colunas (`ft_avaliacoes_pedidos`)

A tabela de fatos **`ft_avaliacoes_pedidos`** foi transformada da camada **Bronze** para a **Silver**, com foco primário na **limpeza de dados** para garantir a integridade referencial e a validade temporal dos registros.

---

### **1. Regras de Qualidade e Limpeza de Dados**

Foi implementado um filtro rigoroso para remover registros inválidos, visando a qualidade dos dados na camada Silver:

* **Validação de `id_pedido` (Chave Externa):**
    * **ID Incorreto:** Registros onde o `id_pedido` é **nulo** ou tem menos de 1 caractere

* **Validação de Datas (`review_creation_date` e `review_answer_timestamp`):**
    * **Data Nula:** Registros onde `data_comentario` ou `data_resposta` são **nulos**.
    * **Data Futura/Fora de Escopo:** Registros onde `data_comentario` ou `data_resposta` são **posteriores à data de execução da transformação** ou fora do escopo histórico esperado do dataset.
    
* **8850 linhas foram removidas**
---

### **2.  Mapeamento e Renomeação de Colunas**

| Coluna na Camada Bronze | Coluna na Camada Silver |
| :---------------------- | :---------------------- |
| `review_id` | **`id_avaliacao`** |
| `order_id` | **`id_pedido`** |
| `review_score` | **`avaliacao`** |
| `review_comment_title` | **`titulo_comentario`** |
| `review_comment_message` | **`comentario`** |
| `review_creation_date` | **`data_comentario`** |
| `review_answer_timestamp` | **`data_resposta`** |

In [0]:
avaliacoes_silver = (
    spark.table("medalhao.bronze.ft_avaliacoes_pedidos")
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.col("review_score").alias("avaliacao"),
        F.col("review_comment_title").alias("titulo_comentario"),
        F.col("review_comment_message").alias("comentario"),

        # usei try_to_timestamp do sql que vai converter qualquer data diferente do formato 'yyyy-MM-dd HH:mm:ss' para null
        F.expr("try_to_timestamp(review_creation_date, 'yyyy-MM-dd HH:mm:ss')").alias("data_comentario"),
        F.expr("try_to_timestamp(review_answer_timestamp, 'yyyy-MM-dd HH:mm:ss')").alias("data_resposta"),

        F.col("ingestion_timestamp").alias("data_ingestao_bronze")
    )
)

total_inicial = avaliacoes_silver.count()

# condições de id_pedido válido
# 1 - não ser nulo
# 2 - ter pelo menos 1 caractere
cond_id_valido = (
    F.col("id_pedido").isNotNull() &
    (F.length(F.col("id_pedido")) > 0)
)

# condições de data_comentario e data_resposta válidos
# 1 - não ser nulo(lembrando que datas com texto/inválidas vão ser consideradas nulo aqui por conta do try_to_timestamp)
# 2 - ser menor ou igual ao timestamp atual
cond_data_valida = (
    F.col("data_comentario").isNotNull() &
    F.col("data_resposta").isNotNull() &
    (F.col("data_comentario") <= F.current_timestamp()) &
    (F.col("data_resposta") <= F.current_timestamp())
)


avaliacoes_silver_filtrado = avaliacoes_silver.filter(cond_id_valido & cond_data_valida)

total_final = avaliacoes_silver_filtrado.count()
linhas_removidas = total_inicial - total_final

print(f"Total de registros removidos: {linhas_removidas}")

avaliacoes_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_avaliacoes_pedidos")

avaliacoes_silver.display()


## **6. Renomeação de Colunas (`ft_produtos`)**

| Coluna na Camada Bronze | Coluna na Camada Silver |
| :---------------------- | :---------------------- |
| `product_id` | **`id_produto`** |
| `product_category_name` | **`categoria_produto`** |
| `product_weight_g` | **`peso_produto_gramas`** |
| `product_length_cm` | **`comprimento_centimetros`** |
| `product_height_cm` | **`altura_centimetros`** |
| `product_width_cm` | **`largura_centimetros`** |


In [0]:
produtos_silver = (
    spark.table("medalhao.bronze.ft_produtos")
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_weight_g").alias("peso_produto_gramas"),
        F.col("product_length_cm").alias("comprimento_centimetros"),
        F.col("product_height_cm").alias("altura_centimetros"),
        F.col("product_width_cm").alias("largura_centimetros"),
        F.col("ingestion_timestamp").alias("data_ingestao_bronze")  # mantém o histórico da Bronze
    )
)

produtos_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_produtos")

produtos_silver.display()

## 7. Mapeamento e Renomeação de Colunas de (`ft_vendedores`) 

| Coluna na Camada Bronze | Coluna na Camada Silver |
| :---------------------- | :---------------------- |
| `seller_id` | **`id_vendedor`** |
| `seller_zip_code_prefix` | **`prefixo_cep`** |
| `seller_city` | **`cidade`** |
| `seller_state` | **`estado`** |


In [0]:
vendedores_silver = (
    spark.table("medalhao.bronze.ft_vendedores")
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado"),
        F.col("ingestion_timestamp").alias("data_ingestao_bronze")
    )
)

vendedores_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.ft_vendedores")

vendedores_silver.display()


## **8.  Mapeamento e Renomeação de Colunas (`dm_categoria_produtos_traducao`)**

| Coluna na Camada Bronze | Coluna na Camada Silver |
| :---------------------- | :---------------------- |
| `product_category_name` | **`nome_produto_pt`** |
| `product_category_name_english` | **`nome_produto_en`** |

In [0]:
categoria_produtos_traducao_silver = (
    spark.table("medalhao.bronze.dm_categoria_produtos_traducao")
    .select(
        F.col("product_category_name").alias("nome_produto_pt"),
        F.col("product_category_name_english").alias("nome_produto_en"),
        F.col("ingestion_timestamp").alias("data_ingestao_bronze")
    )
)

categoria_produtos_traducao_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.dm_categoria_produtos_traducao")

categoria_produtos_traducao_silver.display()


## **9.  Mapeamento e Renomeação de Colunas (`dm_cotacao_dolar`)**

---

### **1.  Regras de Qualidade e Tratamento de Dados**

* **Preenchimento de Cotações Faltantes (Finais de Semana):** A API de cotação não fornece dados para finais de semana. Para garantir a continuidade da série temporal, as cotações faltantes (Sábado e Domingo) são preenchidas com o **valor de fechamento da cotação da Sexta-feira anterior**.


---

### **2.  Mapeamento e Renomeação de Colunas**

| Coluna na Camada Bronze | Coluna na Camada Silver |
| :---------------------- | :---------------------- |
| `cotacaoCompra` | **`cotacao_dolar`** |
| `dataHoraCotacao` | **`data`** |

In [0]:

# capturando a data no formato yyyy-MM-dd HH:mm:ss.SSS e depois transformando para o formato yyyy-MM-dd
cotacao_silver = (
    spark.table("medalhao.bronze.dm_cotacao_dolar")
    .select(
        F.col("cotacaoCompra").cast("decimal(12,2)").alias("cotacao_dolar"),
        F.expr("try_to_timestamp(dataHoraCotacao, 'yyyy-MM-dd HH:mm:ss.SSS')").alias("data_hora"),
    )
    .filter(F.col("data_hora").isNotNull())
    .withColumn("data", F.to_date(F.col("data_hora")))
)

# como eu pretendo fazer uma lista com todas as datas do intervalo do cotacao_silver
# vou capturar a data minima e maxima primeiramente
min_max_dates = cotacao_silver.select(
    F.min(F.col("data")).alias("start_date"),
    F.max(F.col("data")).alias("end_date")
).collect()[0]

start_date = min_max_dates["start_date"]
end_date = min_max_dates["end_date"]


# aqui faço uma tabela com todas as datas do nosso intervalo [start_date, end_date]
df_full_dates = spark.range(1).select(
    F.explode(                                                                              # o F.explode faz a tabela
        F.expr(f"sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)") # o F.expr faz a lista
    ).alias("data_final")
)

# left  join entre todas as datas e o cotacao_silver, basicamente vamos inserir os sabados e domingos com valores NULL na cotacao
df_gapped = df_full_dates.join(
    cotacao_silver,
    df_full_dates["data_final"] == cotacao_silver["data"],
    "left" 
).select(
    F.col("cotacao_dolar"),
    F.col("data_final").alias("data"),
)

# definir a janela de Forward Fill: ordena por data e olha do início da partição
window_fill = (
    Window.orderBy(F.col("data"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# usando last_value passando True como parametro, quando encontrarmos um valor Null em cotacao_dolar, ele vai olhar para
# a última partição que não é NULL e vai propagar esse valor para o próximo dia
# garantindo que o valor de sexta-feira seja propagado para Sábado e Domingo.
cotacao_silver_final = df_gapped.withColumn(
    "cotacao_dolar",
    F.last_value(F.col("cotacao_dolar"), True).over(window_fill)
)

cotacao_silver_final.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{silver_db_name}.dm_cotacao_dolar")

cotacao_silver_final.display()

## **Validação Final**

In [0]:
pedidos_silver = spark.table("medalhao.silver.ft_pedidos")
consumidores_silver = spark.table("medalhao.silver.ft_consumidores")

# pedidos órfãos = existem em pedidos, mas não em consumidores
pedidos_orfaos = pedidos_silver.join(
    consumidores_silver,
    pedidos_silver["id_consumidor"] == consumidores_silver["id_consumidor"],
    "left_anti"
)

qtd_pedidos_orfaos = pedidos_orfaos.count()
print(f"Pedidos órfãos : {qtd_pedidos_orfaos}")

# se houverem órfãos, remove
pedidos_silver = pedidos_silver.join(
    consumidores_silver,
    ["id_consumidor"],
    "inner"
)

itens_silver = spark.table("medalhao.silver.ft_itens_pedidos")

# itens órfãos = existem em itens, mas não em pedidos
itens_orfaos = itens_silver.join(
    pedidos_silver,
    itens_silver["id_pedido"] == pedidos_silver["id_pedido"],
    "left_anti"
)

qtd_itens_orfaos = itens_orfaos.count()
print(f"Itens órfãos: {qtd_itens_orfaos}")

# se houverem órfãos, remove
itens_silver = itens_silver.join(
    pedidos_silver,
    ["id_pedido"],
    "inner"
)

# --------------------------------------------------------------------

pedidos = spark.table("medalhao.silver.ft_pedidos")
consumidores = spark.table("medalhao.silver.ft_consumidores")
pagamentos = spark.table("medalhao.silver.ft_pagamentos_pedidos")
cotacao = spark.table("medalhao.silver.dm_cotacao_dolar")

# agregando todo valor pago em cada pedido
pagamentos_agg = pagamentos.groupBy("id_pedido").agg(
    F.sum("valor_pagamento").alias("valor_total_pago_brl")
)

pedido_total = (
    pedidos
    .join(consumidores, "id_consumidor", "inner")
    .join(pagamentos_agg, "id_pedido", "left")
    .join(
        cotacao,
        F.to_date(pedidos["pedido_compra_timestamp"]) == cotacao["data"],
        "left"
    )
    .select(
        F.to_date(pedidos["pedido_compra_timestamp"]).alias("data_pedido"),
        pedidos["id_pedido"],
        pedidos["id_consumidor"],
        pedidos["status"],
        F.round(F.col("valor_total_pago_brl"), 2).alias("valor_total_pago_brl"),
        F.round(
            F.col("valor_total_pago_brl") / F.col("cotacao_dolar"), 2
        ).alias("valor_total_pago_usd")
    )
)


pedido_total.write.mode("overwrite").format("delta").saveAsTable("medalhao.silver.ft_pedido_total")

pedido_total.orderBy("data_pedido").display()

In [0]:
%sql 
SELECT 
    i.id_pedido,
    i.id_produto,
    i.preco_BRL,
    p.status,
    pg.forma_pagamento,
    pg.valor_pagamento
FROM medalhao.silver.ft_itens_pedidos AS i
LEFT JOIN medalhao.silver.ft_pedidos AS p
    ON i.id_pedido = p.id_pedido
LEFT JOIN medalhao.silver.ft_pagamentos_pedidos AS pg
    ON i.id_pedido = pg.id_pedido
WHERE i.id_pedido = 'bfbd0f9bdef84302105ad712db648a6c';


In [0]:
# caso haja necessidade de exclusão das tabelas

''' spark.sql("DROP TABLE IF EXISTS medalhao.silver.dm_categoria_produtos_traducao")
spark.sql("DROP TABLE IF EXISTS medalhao.silver.dm_cotacao_dolar")
spark.sql("DROP TABLE IF EXISTS medalhao.silver.ft_consumidores")
spark.sql("DROP TABLE IF EXISTS medalhao.silver.ft_itens_pedidos")
spark.sql("DROP TABLE IF EXISTS medalhao.silver.ft_pagamentos_pedidos")
spark.sql("DROP TABLE IF EXISTS medalhao.silver.ft_pedidos")
spark.sql("DROP TABLE IF EXISTS medalhao.silver.ft_produtos")
spark.sql("DROP TABLE IF EXISTS medalhao.silver.ft_vendedores")
spark.sql("DROP TABLE IF EXISTS medalhao.silver.ft_avaliacoes_pedidos")
spark.sql("DROP TABLE IF EXISTS medalhao.silver.ft_pedido_total") '''